In [74]:
import os
import io
import sys
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

import lightning as L
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint

import wandb
import torchmetrics

from sklearn.metrics import roc_auc_score
from new_data.data_loader import train_df, val_df
import cv2

In [75]:
sys.path.append("/Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA")
from lib.OvaMTA_seg import TransRaUNet_CLF_xiaorong as OvaSegModel
from lib.OvaMTA_diag import TransRaUNet_CLF_xiaorong as OvaDiagModel

from utils.smooth_l1_loss import SmoothL1Loss


In [76]:
def build_stage2_dataframe(df, model_seg, device):
    model_seg.eval()
    rows = [] #lista dove costruisci il nuovo dataframe 

    transform = transforms.Compose([
    transforms.Resize((352, 352)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    for i in range(len(df)):
        row = df.iloc[i] #ora è una riga del dataframe

        image = Image.open(io.BytesIO(row["image"])).convert("RGB")

        x = transform(image).unsqueeze(0).to(device) #x input del modello, batch size=1

        with torch.no_grad():
            out5, out4, out3, out2, cls_out, _ = model_seg(x)

            res = out5 + out4 + out3 + out2
            mask_pred = (res[0,0] > 0).cpu().numpy().astype(np.uint8) #res[0,0] vuol dire prima immgine primo canale 

        # trova bbox
        contours, _ = cv2.findContours(mask_pred, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

        if len(contours) == 0:
            x_min, y_min, x_max, y_max = 0, 0, image.size[0], image.size[1]
            has_roi = False

        else:
            c = max(contours, key=cv2.contourArea)
            x_352, y_352, w_352, h_352 = cv2.boundingRect(c)

        orig_w, orig_h = image.size

        # margine come nella repository: +/- 10 pixel nello spazio 352x352
        margin_352 = 10

        x1_352 = max(0, x_352 - margin_352)
        y1_352 = max(0, y_352 - margin_352)
        x2_352 = min(352, x_352 + w_352 + margin_352)
        y2_352 = min(352, y_352 + h_352 + margin_352)

        scale_x = orig_w / 352
        scale_y = orig_h / 352

        x_min = int(x1_352 * scale_x)
        y_min = int(y1_352 * scale_y)
        x_max = int(x2_352 * scale_x)
        y_max = int(y2_352 * scale_y)

        x_min = max(0, x_min)
        y_min = max(0, y_min)
        x_max = min(orig_w, x_max)
        y_max = min(orig_h, y_max)

        if x_max <= x_min or y_max <= y_min:
            x_min, y_min, x_max, y_max = 0, 0, orig_w, orig_h
            has_roi = False
        else:
            has_roi = True
        new_row = row.to_dict() #converte quella riga in un dizionario
        new_row["crop_xmin"] = x_min
        new_row["crop_ymin"] = y_min
        new_row["crop_xmax"] = x_max
        new_row["crop_ymax"] = y_max
        new_row["has_roi"] = has_roi


            
        rows.append(new_row)

    return pd.DataFrame(rows) #nuovo dataframe

In [77]:
class OvaDiagWrapperDataset(data.Dataset):
    def __init__(self, df, trainsize=352):
        self.df = df.reset_index(drop=True)
        self.trainsize = trainsize
        

        self.img_transform = transforms.Compose([
            transforms.Resize((trainsize, trainsize)),
            transforms.ToTensor(),
            transforms.Normalize(
                [0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225]
            )
        ])

        self.gt_transform = transforms.Compose([
            transforms.Resize(
                (trainsize, trainsize),
                interpolation=transforms.InterpolationMode.NEAREST
            ),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.df)
    



    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(io.BytesIO(row["image"])).convert("RGB")
        mask = Image.open(io.BytesIO(row["mask"])).convert("L")

        mask_np = np.array(mask)
        mask_bin = (mask_np > 0).astype(np.uint8)

        x_min = int(row["crop_xmin"])
        y_min = int(row["crop_ymin"])
        x_max = int(row["crop_xmax"])
        y_max = int(row["crop_ymax"])

        image_patch = image.crop((x_min, y_min, x_max, y_max))
        mask_patch = Image.fromarray(mask_bin * 255).crop((x_min, y_min, x_max, y_max))

        image_patch = self.img_transform(image_patch)
        gt_patch = self.gt_transform(mask_patch)
        gt_patch = (gt_patch > 0).float()

        label = torch.tensor(row["risk_class"], dtype=torch.long)

        return image_patch, gt_patch, label

In [78]:


os.chdir("/Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seg_ckpt_path = "test_ovamta_seg/s148uznn/checkpoints/epoch=1-step=28.ckpt" #checkpoint migliori primo modello

# sarebbe il modello "puro"
model_seg = OvaSegModel(training=False).to(device)

# carica checkpoint Lightning
ckpt = torch.load(seg_ckpt_path, map_location=device)

# i pesi del modello Lightning stanno in ckpt["state_dict"]
state_dict = ckpt["state_dict"] #state_dict è un dizionario che contiene tutti i pesi del modello

# nel checkpoint Lightning le chiavi iniziano con "model."
# ma il modello puro OvaMTA non ha quel prefisso, quindi lo togliamo
state_dict = {k.replace("model.", ""): v for k, v in state_dict.items()}

model_seg.load_state_dict(state_dict)
model_seg.eval()

print("Stage-1 checkpoint loaded correctly.")

Stage-1 checkpoint loaded correctly.


In [79]:
train_df_diag = build_stage2_dataframe(train_df, model_seg, device)
val_df_diag = build_stage2_dataframe(val_df, model_seg, device)

train_dataset_diag = OvaDiagWrapperDataset(train_df_diag, trainsize=352)
val_dataset_diag = OvaDiagWrapperDataset(val_df_diag, trainsize=352)

train_loader_diag = DataLoader(train_dataset_diag, batch_size=8, shuffle=True)
val_loader_diag = DataLoader(val_dataset_diag, batch_size=8, shuffle=False)

In [80]:
batch = next(iter(train_loader_diag))
images, gts, labels = batch

print(images.shape)
print(gts.shape)
print(labels.shape)
print(torch.unique(gts))
print(torch.unique(labels))


torch.Size([8, 3, 352, 352])
torch.Size([8, 1, 352, 352])
torch.Size([8])
tensor([0., 1.])
tensor([0, 1])


In [81]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model_diag = TransRaUNet_CLF_xiaorong(training=True).to(device)

# x = torch.randn(2, 3, 352, 352).to(device)
# out = model_diag(x)

# print(type(out))
# print(len(out))

# for i, elem in enumerate(out):
#     if torch.is_tensor(elem):
#         print(i, elem.shape)
#     else:
#         print(i, type(elem))

In [82]:
def structure_loss(pred, mask):
    weit = 1 + 5 * torch.abs(
        F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask
    )
    wbce = F.binary_cross_entropy_with_logits(pred, mask, reduction="none")
    wbce = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3))

    pred = torch.sigmoid(pred)
    inter = ((pred * mask) * weit).sum(dim=(2, 3))
    union = ((pred + mask) * weit).sum(dim=(2, 3))
    wiou = 1 - (inter + 1) / (union - inter + 1)

    return (wbce + wiou).mean()

In [83]:
class LitOvaDiag(L.LightningModule):
    def __init__(self, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()

        self.model = OvaDiagModel(training=True)

        cls_weights = torch.tensor([1.397, 0.779], dtype=torch.float)
        self.register_buffer("cls_weights", cls_weights)

        self.cls_ce = nn.CrossEntropyLoss(weight=self.cls_weights)
        self.cls_reg = SmoothL1Loss()

        self.train_cls_f1 = torchmetrics.classification.BinaryF1Score()
        self.val_cls_f1 = torchmetrics.classification.BinaryF1Score()

        self.train_cls_precision = torchmetrics.classification.BinaryPrecision()
        self.val_cls_precision = torchmetrics.classification.BinaryPrecision()

        self.train_cls_recall = torchmetrics.classification.BinaryRecall()
        self.val_cls_recall = torchmetrics.classification.BinaryRecall()

        self.train_cls_auc = torchmetrics.AUROC(task="binary")
        self.val_cls_auc = torchmetrics.AUROC(task="binary")

        self.train_seg_dice = torchmetrics.classification.BinaryF1Score()
        self.val_seg_dice = torchmetrics.classification.BinaryF1Score()

        self.train_seg_iou = torchmetrics.classification.BinaryJaccardIndex()
        self.val_seg_iou = torchmetrics.classification.BinaryJaccardIndex()

    def forward(self, x):
        return self.model(x)

    def compute_loss(self, images, gts, labels):
        out5, out4, out3, out2, cls_out, features = self.model(images)

        loss5 = structure_loss(out5, gts)
        loss4 = structure_loss(out4, gts)
        loss3 = structure_loss(out3, gts)
        loss2 = structure_loss(out2, gts)

        
        weight = self.cls_weights
        loss1 = self.cls_ce(cls_out, labels) + self.cls_reg(cls_out, labels, weight=weight)

        seg_loss = loss2 + loss3 + loss4 + loss5
        loss = loss1 + seg_loss

        return loss, loss1, loss2, loss3, loss4, loss5, cls_out, out5, out4, out3, out2

    def training_step(self, batch, batch_idx):
        images, gts, labels = batch

        loss, loss1, loss2, loss3, loss4, loss5, cls_out, out5, out4, out3, out2 = self.compute_loss(
            images, gts, labels
        )

        seg_loss = loss2 + loss3 + loss4 + loss5

        seg_logits = out5 + out4 + out3 + out2
        seg_preds = (seg_logits > 0).long()
        gts_int = gts.long()

        cls_preds = torch.argmax(cls_out, dim=1)
        cls_acc = (cls_preds == labels).float().mean()

        cls_probs = torch.softmax(cls_out, dim=1)[:, 1]

        train_seg_iou = self.train_seg_iou(seg_preds, gts_int)
        train_cls_auc = self.train_cls_auc(cls_probs, labels)

        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train_cls_loss", loss1, on_step=True, on_epoch=True)
        self.log("train_seg_loss", seg_loss, on_step=True, on_epoch=True)

        self.log("train_seg_loss2", loss2, on_step=True, on_epoch=True)
        self.log("train_seg_loss3", loss3, on_step=True, on_epoch=True)
        self.log("train_seg_loss4", loss4, on_step=True, on_epoch=True)
        self.log("train_seg_loss5", loss5, on_step=True, on_epoch=True)

        self.log("train_cls_acc", cls_acc, prog_bar=True, on_step=True, on_epoch=True)

        self.log("train_seg_dice", self.train_seg_dice(seg_preds, gts_int), prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_seg_iou", train_seg_iou, prog_bar=True, on_step=False, on_epoch=True)

        self.log("train_cls_f1", self.train_cls_f1(cls_preds, labels), prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_cls_precision", self.train_cls_precision(cls_preds, labels), on_step=False, on_epoch=True)
        self.log("train_cls_recall", self.train_cls_recall(cls_preds, labels), on_step=False, on_epoch=True)
        self.log("train_cls_auc", train_cls_auc, prog_bar=True, on_step=False, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        images, gts, labels = batch

        loss, loss1, loss2, loss3, loss4, loss5, cls_out, out5, out4, out3, out2 = self.compute_loss(
            images, gts, labels
        )

        seg_loss = loss2 + loss3 + loss4 + loss5

        seg_logits = out5 + out4 + out3 + out2
        seg_preds = (seg_logits > 0).long()
        gts_int = gts.long()

        cls_preds = torch.argmax(cls_out, dim=1)
        cls_acc = (cls_preds == labels).float().mean()

        cls_probs = torch.softmax(cls_out, dim=1)[:, 1]

        self.val_cls_auc.update(cls_probs, labels)
        self.val_cls_f1.update(cls_preds, labels)
        self.val_cls_precision.update(cls_preds, labels)
        self.val_cls_recall.update(cls_preds, labels)

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_cls_loss", loss1, on_step=False, on_epoch=True)
        self.log("val_seg_loss", seg_loss, on_step=False, on_epoch=True)

        self.log("val_seg_loss2", loss2, on_step=False, on_epoch=True)
        self.log("val_seg_loss3", loss3, on_step=False, on_epoch=True)
        self.log("val_seg_loss4", loss4, on_step=False, on_epoch=True)
        self.log("val_seg_loss5", loss5, on_step=False, on_epoch=True)

        self.log("val_cls_acc", cls_acc, prog_bar=True, on_step=False, on_epoch=True)

        self.log("val_seg_dice", self.val_seg_dice(seg_preds, gts_int), prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_seg_iou", self.val_seg_iou(seg_preds, gts_int), prog_bar=True, on_step=False, on_epoch=True)

        return loss

    def on_validation_epoch_end(self):
        self.log("val_cls_auc", self.val_cls_auc.compute(), prog_bar=True)
        self.log("val_cls_f1", self.val_cls_f1.compute(), prog_bar=True)
        self.log("val_cls_precision", self.val_cls_precision.compute())
        self.log("val_cls_recall", self.val_cls_recall.compute())


        self.val_cls_auc.reset()
        self.val_cls_f1.reset()
        self.val_cls_precision.reset()
        self.val_cls_recall.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr
        )
        return optimizer

In [84]:
def train_diag():
    wandb.init()
    lr = wandb.config.lr

    wandb_logger = WandbLogger(
        project="test_ovamta_diag",
        log_model=False
    )

    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        mode="min",
        save_top_k=1
    )

    model_lightning_diag = LitOvaDiag(
        lr=lr
    )

    trainer = L.Trainer(
        max_epochs=5,
        logger=wandb_logger,
        callbacks=[checkpoint_callback],
        log_every_n_steps=1
    )

    trainer.fit(model_lightning_diag, train_dataloaders=train_loader_diag, val_dataloaders=val_loader_diag)

    wandb.finish()

In [85]:
sweep_config = {
    "method": "grid",
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "lr": {"values": [1e-3, 1e-4]}
    }
}

sweep_id = wandb.sweep(sweep_config, project="test_ovamta_diag")
wandb.agent(sweep_id, function=train_diag, count=2)

Create sweep with ID: l4xqj3oh
Sweep URL: https://wandb.ai/sara-tramontana02-/test_ovamta_diag/sweeps/l4xqj3oh


wandb: Agent Starting Run: ukagm81m with config:
wandb: 	lr: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/saratramontana/.netrc.


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ model               │ TransRaUNet_CLF_xiaorong │ 29.8 M │ train │     0 │
│ 1  │ cls_ce              │ CrossEntropyLoss         │      0 │ train │     0 │
│ 2  │ cls_reg             │ SmoothL1Loss             │      0 │ train │     0 │
│ 3  │ train_cls_f1        │ BinaryF1Score            │      0 │ train │     0 │
│ 4  │ val_cls_f1          │ BinaryF1Score            │      0 │ train │     0 │
│ 5  │ train_cls_precision │ BinaryPrecision          │      0 │ train │     0 │
│ 6  │ val_cls_precision   │ BinaryPrecision          │      0 │ train │     0 │
│ 7  │ train_cls_recall    │ BinaryRecall             │      0 │ train │     0 │
│ 8  │ val_cls_recall      │ BinaryRecall             │      0 │ train │     0 │
│ 9  │ train_cls_auc       │ BinaryAUROC              │      0 │ train │     0 │
│ 10 │ val_cls_auc         │ BinaryAUROC              │      0 │ train │     0 │
│ 11 │ train_seg_dice      │ BinaryF1Score            │      0 │ train │     0 │
│ 12 │ val_seg_dice        │ BinaryF1Score            │      0 │ train │     0 │
│ 13 │ train_seg_iou       │ BinaryJaccardIndex       │      0 │ train │     0 │
│ 14 │ val_seg_iou         │ BinaryJaccardIndex       │      0 │ train │     0 │
└────┴─────────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 29.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 29.8 M                                                                                               
Total estimated model params size (MB): 119                                                                        
Modules in train mode: 619                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/dat
a_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing 
the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in 
false positive score
  warnings.warn(*args, **kwargs)

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/dat
a_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=5` reached.


epoch,▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆████████
train_cls_acc_epoch,▁████
train_cls_acc_step,▅▅▁█████▆██████▆██████▆████▆█████████▆█▅
train_cls_auc,▁▂█▄▆
train_cls_f1,▁███▆
train_cls_loss_epoch,█▂▂▁▁
train_cls_loss_step,▄▃▃▇▁▁▇▁▁█▁▁▁▁▁█▁▁▁▁▁▇▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁█▇
train_cls_precision,▁████
train_cls_recall,▇▇▇█▁
train_loss_epoch,█▅▄▂▁
+28,...


In [86]:
import torch
import glob

ckpts = glob.glob(
    "/Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA/test_ovamta_diag/*/checkpoints/*.ckpt"
)

for path in ckpts:
    ckpt = torch.load(path, map_location="cpu")
    print("\n", path)
    print("epoch:", ckpt.get("epoch"))
    print("global_step:", ckpt.get("global_step"))
    print("callbacks keys:")
    for k in ckpt.get("callbacks", {}).keys():
        print("  ", k)


 /Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA/test_ovamta_diag/rd4fdern/checkpoints/epoch=0-step=5.ckpt
epoch: 0
global_step: 5
callbacks keys:
   ModelCheckpoint{'monitor': 'val_loss', 'mode': 'min', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}

 /Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA/test_ovamta_diag/7zb65y1i/checkpoints/epoch=0-step=14.ckpt
epoch: 0
global_step: 14
callbacks keys:
   ModelCheckpoint{'monitor': 'val_loss', 'mode': 'min', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}

 /Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA/test_ovamta_diag/x6evxkcw/checkpoints/epoch=8-step=126.ckpt
epoch: 8
global_step: 126
callbacks keys:
   ModelCheckpoint{'monitor': 'val_loss', 'mode': 'min', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}

 /Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA/test_

In [87]:
for path in ckpts:
    ckpt = torch.load(path, map_location="cpu")
    print("\n", path)

    callbacks = ckpt.get("callbacks", {})
    for name, cb in callbacks.items():
        if "best_model_score" in cb:
            print("monitor:", cb.get("monitor"))
            print("best_model_score:", cb.get("best_model_score"))
            print("best_model_path:", cb.get("best_model_path"))


 /Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA/test_ovamta_diag/rd4fdern/checkpoints/epoch=0-step=5.ckpt
monitor: val_loss
best_model_score: tensor(5.8528)
best_model_path: ./test_ovamta_diag/rd4fdern/checkpoints/epoch=0-step=5.ckpt

 /Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA/test_ovamta_diag/7zb65y1i/checkpoints/epoch=0-step=14.ckpt
monitor: val_loss
best_model_score: tensor(7.7117)
best_model_path: ./test_ovamta_diag/7zb65y1i/checkpoints/epoch=0-step=14.ckpt

 /Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA/test_ovamta_diag/x6evxkcw/checkpoints/epoch=8-step=126.ckpt
monitor: val_loss
best_model_score: tensor(3.9697)
best_model_path: ./test_ovamta_diag/x6evxkcw/checkpoints/epoch=8-step=126.ckpt

 /Users/saratramontana/Documents/test_segmentation_model/external/OvaMTA/test_ovamta_diag/aogh062w/checkpoints/epoch=2-step=42.ckpt
monitor: val_loss
best_model_score: tensor(5.0769)
best_model_path: ./test_ovamta